In [1]:
import pandas as pd
import clickhouse_connect

In [12]:
client = clickhouse_connect.get_client(host='localhost', username='etl_user', password='')

In [13]:
client.command("""
CREATE TABLE IF NOT EXISTS electronics_events (
    event_time DateTime,
    event_type LowCardinality(String),
    product_id UInt32,
    category_id UInt64,
    category_code LowCardinality(String),
    brand LowCardinality(String),
    price Float32,
    user_id UInt32,
    user_session String
) ENGINE = MergeTree()
ORDER BY (event_time, user_id)
""")

In [14]:
files = ['dataset/2019-Oct.csv', 'dataset/2019-Nov.csv']

for file in files:
    print(f"Обработка файла: {file}")
    
    reader = pd.read_csv(file, chunksize=1000000)
    
    for i, chunk in enumerate(reader):
        chunk['event_time'] = pd.to_datetime(chunk['event_time'].str.replace(' UTC', ''), format='%Y-%m-%d %H:%M:%S') 
        chunk = chunk.fillna('unknown')
        
        chunk['price'] = chunk['price'].astype('float32')
        chunk['product_id'] = chunk['product_id'].astype('uint32')
        chunk['user_id'] = chunk['user_id'].astype('uint32')
        
        client.insert('electronics_events', chunk)
        
        if i % 10 == 0:
            print(f"Загружено {i} млн строк из текущего файла")

print("Данные успешно загружены")

Обработка файла: dataset/2019-Oct.csv
Загружено 0 млн строк из текущего файла
Загружено 10 млн строк из текущего файла
Загружено 20 млн строк из текущего файла
Загружено 30 млн строк из текущего файла
Загружено 40 млн строк из текущего файла
Обработка файла: dataset/2019-Nov.csv
Загружено 0 млн строк из текущего файла
Загружено 10 млн строк из текущего файла
Загружено 20 млн строк из текущего файла
Загружено 30 млн строк из текущего файла
Загружено 40 млн строк из текущего файла
Загружено 50 млн строк из текущего файла
Загружено 60 млн строк из текущего файла
Данные успешно загружены
